In [ ]:
"""
Gaussian Process Regression (GPR)
Flexural Strength Prediction – Gypsum-Based Composites

Author: Haseeb Ahmad
Description:
This script trains a GPR model, optimizes kernel hyperparameters
via multiple restarts, evaluates performance, and generates:
    • LML convergence plot
    • Actual vs Predicted plot
    • 95% Confidence interval plot
    • Permutation Feature Importance
    • SHAP analysis
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# ==============================
# Configuration
# ==============================

DATA_PATH = "../data/gypsum_data.csv"
FIGURE_PATH = "../figures/gpr_flexural"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_RESTARTS = 10

os.makedirs(FIGURE_PATH, exist_ok=True)
np.random.seed(RANDOM_STATE)


# ==============================
# Load Data
# ==============================

def load_data(path):

    df = pd.read_csv(path, header=2)

    X = df.iloc[:, [4, 5, 6, 7, 8, 9, 10]].astype(float)
    y = df.iloc[:, 12].astype(float)

    data = pd.concat([X, y], axis=1).dropna()

    X_clean = data.iloc[:, :-1].values
    y_clean = data.iloc[:, -1].values

    return X_clean, y_clean


# ==============================
# Kernel Optimization
# ==============================

def optimize_gpr_kernel(X_train, y_train):

    base_kernel = (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
        + WhiteKernel(noise_level=0.1)
    )

    best_lml = -np.inf
    best_model = None
    lml_values = []

    print("\nOptimizing GPR Kernel Across Restarts...\n")

    for i in range(N_RESTARTS):

        gpr = GaussianProcessRegressor(
            kernel=base_kernel,
            optimizer="fmin_l_bfgs_b",
            random_state=RANDOM_STATE + i
        )

        gpr.fit(X_train, y_train)

        lml = gpr.log_marginal_likelihood(gpr.kernel_.theta)
        lml_values.append(lml)

        print(f"Restart {i+1}: LML = {lml:.4f}")
        print(f"Kernel: {gpr.kernel_}\n")

        if lml > best_lml:
            best_lml = lml
            best_model = gpr

    return best_model, lml_values


# ==============================
# Evaluation
# ==============================

def evaluate_model(y_true, y_pred):

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print("=" * 50)
    print("GPR MODEL PERFORMANCE")
    print("=" * 50)
    print(f"MSE  : {mse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")
    print("=" * 50)

    return mse, mae, rmse, r2


# ==============================
# Plotting Functions
# ==============================

def plot_lml(lml_values):

    x_vals = range(1, len(lml_values) + 1)

    plt.figure(figsize=(5, 4))
    plt.plot(x_vals, lml_values, marker="o")
    plt.xlabel("Optimizer Restart")
    plt.ylabel("Log-Marginal Likelihood")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/lml_convergence.png", dpi=300)
    plt.show()


def plot_actual_vs_predicted(y_true, y_pred):

    x_vals = np.linspace(y_true.min(), y_true.max(), 200)

    plt.figure(figsize=(5, 4))
    plt.scatter(y_true, y_pred, alpha=0.7, edgecolors="k")
    plt.plot(x_vals, x_vals, "r--", label="Perfect Fit")

    plt.xlabel("Actual Flexural Strength (MPa)")
    plt.ylabel("Predicted Flexural Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/actual_vs_predicted.png", dpi=300)
    plt.show()


def plot_confidence_intervals(y_true, y_pred, sigma):

    indices = np.arange(len(y_true))
    upper = y_pred + 1.96 * sigma
    lower = y_pred - 1.96 * sigma

    plt.figure(figsize=(5, 4))
    plt.plot(indices, y_true, "o", label="Actual")
    plt.plot(indices, y_pred, "--", label="Predicted")
    plt.fill_between(indices, lower, upper, alpha=0.3, label="95% CI")

    plt.xlabel("Sample Index")
    plt.ylabel("Flexural Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/confidence_interval.png", dpi=300)
    plt.show()


def plot_permutation_importance(model, X_test, y_test, feature_names):

    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=20,
        scoring="neg_mean_squared_error",
        random_state=RANDOM_STATE
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/feature_importance.png", dpi=300)
    plt.show()


# ==============================
# SHAP Analysis
# ==============================

def shap_analysis(model, X_train, X_test, feature_names):

    print("\nRunning SHAP KernelExplainer...\n")

    background_size = min(100, X_train.shape[0])
    background = X_train[
        np.random.choice(X_train.shape[0], background_size, replace=False)
    ]

    explainer = shap.KernelExplainer(model.predict, background)
    shap_values = explainer.shap_values(X_test)

    # Summary Plot
    plt.figure()
    shap.summary_plot(shap_values, X_test,
                      feature_names=feature_names,
                      show=False)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/shap_summary.png", dpi=300)
    plt.show()

    return shap_values


# ==============================
# Main
# ==============================

def main():

    # Load data
    X, y = load_data(DATA_PATH)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Optimize kernel
    best_model, lml_values = optimize_gpr_kernel(
        X_train_scaled, y_train
    )

    # Plot LML
    plot_lml(lml_values)

    # Predict
    y_pred, sigma = best_model.predict(
        X_test_scaled, return_std=True
    )

    # Evaluate
    evaluate_model(y_test, y_pred)

    # Plots
    plot_actual_vs_predicted(y_test, y_pred)
    plot_confidence_intervals(y_test, y_pred, sigma)

    feature_names = [
        "Gypsum Strength",
        "Gypsum Quantity",
        "Water Quantity",
        "Water/Gypsum Ratio",
        "Wheat Straw",
        "CaCl2",
        "Ca(OH)2"
    ]

    plot_permutation_importance(
        best_model,
        X_test_scaled,
        y_test,
        feature_names
    )

    shap_analysis(
        best_model,
        X_train_scaled,
        X_test_scaled,
        feature_names
    )


if __name__ == "__main__":
    main()